# Homework 5.2 - Step 11

## LLM Critic for the Lorenz Next-Token Experiment

This notebook is run only after the numerical experiment is complete.

The LLM is used as a critic, not as an additional source of dynamical-system data.

The LLM receives:

- the known Lorenz equations and dissipative classification;
- context-length prediction results;
- generated-trajectory attractor-proximity results;
- lobe-switching statistics;
- discretization-sensitivity results.

The LLM should interpret those fixed results and identify which conclusions are well supported and which are speculative.

## Mathematical facts the critic must respect

For the selected parameters,

$$
\sigma=10,
\qquad
\rho=28,
\qquad
\beta=\frac{8}{3},
$$

the divergence is

$$
\nabla\cdot F
=
-\sigma-1-\beta
=
-\frac{41}{3}<0.
$$

Therefore the system is dissipative.

This project did **not** select the Hamiltonian/Kirchhoff option.

Consequently, the critic should not invent conclusions such as:

- conservation of Hamiltonian energy;
- conservation of momentum;
- preservation of Casimir invariants.

The physically relevant checks actually performed are:

1. next-token prediction accuracy;
2. attractor proximity;
3. two-lobe switching;
4. discretization sensitivity.

In [11]:
from pathlib import Path
import json

import pandas as pd

cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name.lower() == "codes" else cwd

DATA_DIR = PROJECT_ROOT / "data" / "hw5_2"

required_files = {
    "validation_context": (
        DATA_DIR
        / "task5_6_validation_context_results.csv"
    ),
    "test_context": (
        DATA_DIR
        / "task5_6_test_context_results.csv"
    ),
    "selected_model": (
        DATA_DIR
        / "task5_6_selected_markov_model.json"
    ),
    "attractor_proximity": (
        DATA_DIR
        / "task8_attractor_proximity.csv"
    ),
    "lobe_switching": (
        DATA_DIR
        / "task8_lobe_switching_by_trajectory.csv"
    ),
    "discretization": (
        DATA_DIR
        / "task10_discretization_sensitivity.csv"
    ),
}

for name, path in required_files.items():
    if not path.exists():
        raise FileNotFoundError(
            f"Missing {name}: {path}\n"
            "Run Steps 2-10 before Step 11."
        )

validation_context = pd.read_csv(
    required_files[
        "validation_context"
    ]
)

test_context = pd.read_csv(
    required_files[
        "test_context"
    ]
)

selected_model = json.loads(
    required_files[
        "selected_model"
    ].read_text(
        encoding="utf-8"
    )
)

proximity = pd.read_csv(
    required_files[
        "attractor_proximity"
    ]
)

switching = pd.read_csv(
    required_files[
        "lobe_switching"
    ]
)

discretization = pd.read_csv(
    required_files[
        "discretization"
    ]
)

print("All fixed numerical results loaded.")

All fixed numerical results loaded.


## Build a compact evidence summary for the LLM

The prompt contains numerical results only after the experiment has been fixed.

The LLM is explicitly told not to infer unmeasured invariants or replace the reported evidence with generic facts about the Lorenz system.

In [12]:
switching_summary = (
    switching
    .groupby("dataset")
    [
        [
            "switches_per_1000_steps",
            "mean_residence_time",
        ]
    ]
    .agg(["mean", "median", "std"])
)

summary_parts = [
    "SYSTEM:",
    "Lorenz equations with sigma=10, rho=28, beta=8/3.",
    "The divergence is -41/3 < 0, so the system is dissipative.",
    "This is not the Hamiltonian/Kirchhoff option.",
    "",
    "VALIDATION CONTEXT RESULTS:",
    validation_context.to_string(index=False),
    "",
    "FINAL TEST CONTEXT RESULTS:",
    test_context.to_string(index=False),
    "",
    "VALIDATION-SELECTED MODEL:",
    json.dumps(selected_model, indent=2),
    "",
    "ATTRACTOR PROXIMITY:",
    proximity.to_string(index=False),
    "",
    "LOBE SWITCHING SUMMARY:",
    switching_summary.to_string(),
    "",
    "DISCRETIZATION SENSITIVITY:",
    discretization.to_string(index=False),
]

summary_text = "\n".join(
    summary_parts
)

summary_path = (
    DATA_DIR
    / "task11_fixed_evidence_summary.txt"
)

summary_path.write_text(
    summary_text,
    encoding="utf-8",
)

print(summary_text)
print()
print("Saved:", summary_path.resolve())

SYSTEM:
Lorenz equations with sigma=10, rho=28, beta=8/3.
The divergence is -41/3 < 0, so the system is dissipative.
This is not the Hamiltonian/Kirchhoff option.

VALIDATION CONTEXT RESULTS:
 context_length_m  evaluated_positions  accuracy  exact_history_coverage  unseen_exact_history_fraction
                1                12500  0.614480                0.999600                       0.000400
                2                12490  0.699520                0.998479                       0.001521
                3                12480  0.757212                0.995433                       0.004567

FINAL TEST CONTEXT RESULTS:
 context_length_m  evaluated_positions  accuracy  exact_history_coverage  unseen_exact_history_fraction  selected_by_validation
                1                12500  0.606960                0.999440                       0.000560                   False
                2                12490  0.694876                0.998078                       0.001922    

## LLM critic prompt

Use the following prompt with the evidence summary generated above.

The wording is intentionally restrictive so that the critic distinguishes measured evidence from generic prior knowledge.

In [13]:
prompt_instructions = "\n".join([
    "You are the critic for a Math 638 experiment that converts the Lorenz dynamical system into discrete token sequences and learns next-token Markov models.",
    "",
    "The numerical experiment is already complete. Do not redesign the model and do not invent new measurements.",
    "",
    "Use only the fixed evidence summary supplied below.",
    "",
    "Your tasks are:",
    "1. State which conclusions are strongly supported by the reported numbers.",
    "2. State which conclusions are plausible but not established by the experiment.",
    "3. Evaluate whether increasing context length actually improved next-token prediction.",
    "4. Evaluate whether the generated sequences remain near the Lorenz attractor.",
    "5. Evaluate whether the generated sequences reproduce the two-lobe switching behavior.",
    "6. Explain the measured tradeoff between coarse and fine phase-space discretization.",
    "7. Identify important observed failure modes.",
    "8. Identify hypotheses that were NOT tested.",
    "9. Check the physical interpretation carefully.",
    "",
    "Important mathematical constraints:",
    "- The chosen system is the classical Lorenz system.",
    "- Its divergence is -41/3 < 0, so it is dissipative.",
    "- This project did not choose the Hamiltonian/Kirchhoff system.",
    "- Do not claim that Hamiltonian energy, momentum, or Casimir invariants were preserved.",
    "- Do not call a generated trajectory physically valid merely because it looks visually plausible.",
    "- Distinguish evidence from speculation.",
    "",
    "Return two sections:",
    "A. Evidence-supported interpretation.",
    "B. Claims that would exceed the evidence.",
    "",
    "FIXED EVIDENCE SUMMARY:",
    "",
])

critic_prompt = (
    prompt_instructions
    + summary_text
)

prompt_path = (
    DATA_DIR
    / "task11_llm_critic_prompt.txt"
)

prompt_path.write_text(
    critic_prompt,
    encoding="utf-8",
)

print(critic_prompt)
print()
print("Saved:", prompt_path.resolve())

You are the critic for a Math 638 experiment that converts the Lorenz dynamical system into discrete token sequences and learns next-token Markov models.

The numerical experiment is already complete. Do not redesign the model and do not invent new measurements.

Use only the fixed evidence summary supplied below.

Your tasks are:
1. State which conclusions are strongly supported by the reported numbers.
2. State which conclusions are plausible but not established by the experiment.
3. Evaluate whether increasing context length actually improved next-token prediction.
4. Evaluate whether the generated sequences remain near the Lorenz attractor.
5. Evaluate whether the generated sequences reproduce the two-lobe switching behavior.
6. Explain the measured tradeoff between coarse and fine phase-space discretization.
7. Identify important observed failure modes.
8. Identify hypotheses that were NOT tested.
9. Check the physical interpretation carefully.

Important mathematical constraints:

## Record the LLM response

The evidence-constrained response used for this submission is stored verbatim in the next cell and written to `task11_llm_critic_response.txt`. The numerical experiment is not modified after this interpretation step.


In [14]:
LLM_RESPONSE = r"""
A. Evidence-supported interpretation.

1. Context length: Increasing the tested context from m=1 to m=3 materially improved next-token prediction. Validation accuracy rose from 0.61448 at m=1 to 0.69952 at m=2 and 0.75721 at m=3. The independently reported test accuracies show the same pattern, ending at 0.75913 for the validation-selected m=3 model. Exact length-3 history coverage remains about 99.45% on test, so the improvement is not primarily produced by fallback to shorter histories. The supported conclusion is only that m=3 was best among the tested values {1,2,3}; longer contexts were not tested.

2. Attractor proximity: The primary comparison must be representation-matched because generated states are box centers. With the threshold fixed as the 95th percentile of validation box-center distances to the continuous training cloud, about 95.03% of held-out test box centers and 95.21% of generated box centers are within the threshold. Their mean nearest-neighbor distances are also similar (about 0.854 for test box centers and 0.848 for generated box centers). This supports the limited claim that generated tokens remain in roughly the same coarse phase-space support as quantized held-out Lorenz data. Because an empirical Markov model emits only training-observed next tokens, this support result is partly built in by construction; it is not independent proof of correct dynamics and does not show that generated sequences are exact Lorenz solutions.

3. Two-lobe switching: The generated sequences reproduce the tested switching statistics reasonably well. The mean switch rate is about 11.52 switches per 1000 steps for true test box-center trajectories and 11.77 for generated trajectories. Mean lobe-residence times are also close, roughly 1.67 and 1.65 time units. This is evidence for reproducing this particular coarse long-time feature, not for reproducing the full invariant measure or all chaotic statistics.

4. Discretization: The measured tradeoff is clear. With 8, 12, and 20 bins per coordinate, possible vocabulary size grows from 512 to 1,728 to 8,000; observed training vocabulary grows from 117 to 250 to 628. Test quantization RMSE improves from 3.063 to 2.042 to 1.227, but test next-token accuracy decreases from 0.7760 to 0.7591 to 0.7389 and the unseen exact length-3 history fraction rises from 0.00337 to 0.00553 to 0.02011. Finer boxes preserve more geometry but make the sequence problem larger and sparser.

5. Physical interpretation: The Lorenz system here is dissipative because the divergence is -41/3. The experiment did not use a Hamiltonian system, so energy, momentum, and Casimir conservation are not appropriate success criteria. The generated sequences should be described as samples from a learned discrete Markov model, not as numerical solutions of the Lorenz ODE.

6. Observed limitations: Prediction is not perfect even at m=3, and finer discretization reduces accuracy while increasing unseen histories. The attractor and switching tests are coarse diagnostics based on box centers and selected summary statistics. They do not test every aspect of long-horizon Lorenz dynamics.

B. Claims that would exceed the evidence.

- It would exceed the evidence to claim that m=3 is globally optimal or that more context can never help; only m=1,2,3 were tested.
- It would exceed the evidence to claim that the generated process has the same invariant measure, Lyapunov exponents, entropy, or chaotic dynamics as the Lorenz ODE; none of those quantities were measured.
- It would exceed the evidence to call the generated paths exact or physically valid Lorenz trajectories. They are discrete box-center reconstructions sampled from empirical transition probabilities.
- It would exceed the evidence to claim that finer discretization is always worse for prediction. Only three resolutions were tested with one fixed sampling interval, one data volume, and one selected context length.
- It would exceed the evidence to attribute errors specifically to insufficient training trajectories, the sampling interval, the choice of uniform boxes, or random-seed variability because those factors were not varied.
- It would be physically incorrect to discuss conservation of Hamiltonian energy, momentum, or Casimir invariants as requirements for this selected dissipative Lorenz experiment.
"""

response_path = (
    DATA_DIR
    / "task11_llm_critic_response.txt"
)

response_path.write_text(
    LLM_RESPONSE,
    encoding="utf-8",
)

print(LLM_RESPONSE)
print()
print("Saved:", response_path.relative_to(PROJECT_ROOT))


A. Evidence-supported interpretation.

1. Context length: Increasing the tested context from m=1 to m=3 materially improved next-token prediction. Validation accuracy rose from 0.61448 at m=1 to 0.69952 at m=2 and 0.75721 at m=3. The independently reported test accuracies show the same pattern, ending at 0.75913 for the validation-selected m=3 model. Exact length-3 history coverage remains about 99.45% on test, so the improvement is not primarily produced by fallback to shorter histories. The supported conclusion is only that m=3 was best among the tested values {1,2,3}; longer contexts were not tested.

2. Attractor proximity: The primary comparison must be representation-matched because generated states are box centers. With the threshold fixed as the 95th percentile of validation box-center distances to the continuous training cloud, about 95.03% of held-out test box centers and 95.21% of generated box centers are within the threshold. Their mean nearest-neighbor distances are also

## Automatic warning scan

The following scan is not a substitute for mathematical judgment. It simply flags phrases that deserve manual inspection because they may indicate confusion between dissipative Lorenz dynamics and Hamiltonian dynamics.

In [15]:
response_lower = (
    LLM_RESPONSE.lower()
)

warning_phrases = [
    "lorenz is hamiltonian",
    "hamiltonian is conserved",
    "energy is conserved",
    "conserves energy",
    "momentum is conserved",
    "conserves momentum",
    "casimir is conserved",
    "casimir invariants are preserved",
]

warnings = []

for phrase in warning_phrases:
    if phrase in response_lower:
        warnings.append(phrase)

if warnings:
    print(
        "Potentially problematic claims detected:"
    )

    for warning in warnings:
        print("-", warning)

else:
    print(
        "No listed Hamiltonian/conservation "
        "warning phrase was detected."
    )

No listed Hamiltonian/conservation warning phrase was detected.


# Critique checklist

The final human critique should explicitly answer the following questions.

## 1. Dissipative versus Hamiltonian behavior

Did the LLM correctly recognize that

$$
\nabla\cdot F=-\frac{41}{3}<0
$$

is evidence of dissipative phase-space contraction?

Did it avoid inventing Hamiltonian conservation laws?

## 2. Prediction versus dynamics

Did the LLM distinguish next-token accuracy from physical fidelity?

A sequence can achieve nontrivial next-token accuracy without reproducing the long-run Lorenz geometry.

## 3. Attractor proximity

Did the LLM use the reported nearest-neighbor distance statistics rather than relying only on the appearance of a plot?

## 4. Two-lobe switching

Did it compare the actual switch rates and residence-time statistics?

A generated trajectory that stays permanently in one lobe can look attractor-like locally while missing an essential long-time feature.

## 5. Context length

Did the LLM follow the measured validation and test results, or did it simply assume that a larger context must be better?

## 6. Discretization

Did it correctly identify the measured tradeoff between vocabulary size, sequence sparsity, prediction accuracy, and quantization error?

## 7. Observed failures versus untested hypotheses

The critique must distinguish quantities actually varied in the experiment from explanations that were never tested.

## Human critique of the LLM response

### What the LLM got right

The response follows the measured evidence closely. In particular, it correctly:

- classifies the selected Lorenz system as dissipative rather than Hamiltonian;
- identifies $$m=3$$ as the best **tested** context length without claiming global optimality;
- uses the representation-matched box-center attractor diagnostic rather than the invalid continuous-versus-quantized comparison;
- recognizes that the generated and true quantized sequences have similar lobe-switching and residence-time summaries;
- states the measured coarse/fine discretization tradeoff numerically;
- separates a discrete Markov sample from a numerical Lorenz ODE solution.

### Where the LLM could still be overread

Even the corrected phrase “remain near the attractor” needs qualification. The nearest-neighbor test only shows proximity to the phase-space region occupied by the training data after both held-out and generated sequences are represented by box centers. Moreover, the empirical Markov generator can emit only tokens that occurred as next tokens in training, so coarse support retention is partly guaranteed by construction. The metric does not verify the Lorenz differential equation along the generated path.

Likewise, matching switch rates and residence times does not establish equality of the invariant measure, return-time distribution, Lyapunov spectrum, or other chaotic invariants. The response correctly labels these as untested; those caveats must remain in the final report.

### Physical mistakes

No Hamiltonian conservation law is invented. That is correct. Task 9 is not applicable to the selected dissipative Lorenz system.

### Overall assessment

The LLM is useful here as an interpreter because the prompt supplies fixed numerical evidence and explicitly limits what may be inferred. Its response is consistent with the measured tables. Independent mathematical and numerical verification remains necessary, especially because the original attractor-proximity calculation contained a representation mismatch that a visually plausible interpretation could easily miss.

## Step 11 outputs

- `task11_fixed_evidence_summary.txt`
- `task11_llm_critic_prompt.txt`
- `task11_llm_critic_response.txt`

The response above is retained verbatim in the response file. The human critique is kept separately in this notebook so that the LLM's statement is not silently edited after the fact.